In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv("../data/raw/dataset-6ab002b01f9ef078223946.csv")

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sort_values(ascending=False)

In [ ]:
df.value_counts("Internet_Access").sort_index()

In [ ]:
df.value_counts("Exam_Score")

In [ ]:
df[df["Exam_Score"]==101]

In [ ]:
df.isnull().sum()

In [ ]:
teacher_quality_mode = df["Teacher_Quality"].mode()
teacher_quality_mode

In [ ]:
Parental_Education_Level_mode = df["Parental_Education_Level"].mode()
Parental_Education_Level_mode

In [ ]:
Distance_from_Home_mode = df["Distance_from_Home"].mode()
Distance_from_Home_mode

In [ ]:
df.info()

In [ ]:
df[["Hours_Studied","Attendance","Sleep_Hours","Previous_Scores","Tutoring_Sessions","Physical_Activity","Exam_Score"]].mean()

In [ ]:
df[["Hours_Studied","Attendance","Sleep_Hours","Previous_Scores","Tutoring_Sessions","Physical_Activity","Exam_Score"]].median()

In [ ]:
df[["Hours_Studied","Attendance","Sleep_Hours","Previous_Scores","Tutoring_Sessions","Physical_Activity","Exam_Score"]].std()

In [ ]:
df.duplicated().sum()

In [ ]:
plt.plot(df["Exam_Score"])
plt.show()

In [ ]:

sns.histplot(df["Exam_Score"], bins=30, kde=True)
plt.xlabel("Exam_Score")
plt.ylabel("Effectif")
plt.show()

In [ ]:
sns.histplot(df["Hours_Studied"], bins=30, kde=True)
plt.xlabel("Hours_Studied")
plt.ylabel("Effectif")
plt.show()

In [ ]:
df_clean = df.copy()

In [ ]:
cols_mode = ["Teacher_Quality", "Parental_Education_Level", "Distance_from_Home"]

for col in cols_mode:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

In [ ]:
n_dup = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

In [ ]:
sns.boxplot(x=df_clean["Exam_Score"])
plt.title("Boxplot de Exam_Score")
plt.show()

In [ ]:
q1, q3 = df_clean["Exam_Score"].quantile([0.25, 0.75])
iqr = q3 - q1
borne_basse, borne_haute = q1 - 1.5 * iqr, q3 + 1.5 * iqr
est_outlier_iqr = (df_clean["Exam_Score"] < borne_basse) | (df_clean["Exam_Score"] > borne_haute)

In [ ]:
z = stats.zscore(df_clean["Exam_Score"])
est_outlier_z = np.abs(z) > 3

In [ ]:
df_clean = df_clean[~est_outlier_iqr].reset_index(drop=True)

In [ ]:
ordre_lmh = {"Low": 0, "Medium": 1, "High": 2}
mappings_ordinaux = {
    "Parental_Involvement": ordre_lmh,
    "Access_to_Resources": ordre_lmh,
    "Motivation_Level": ordre_lmh,
    "Family_Income": ordre_lmh,
    "Teacher_Quality": ordre_lmh,
    "Parental_Education_Level": {"High School": 0, "College": 1, "Postgraduate": 2},
    "Distance_from_Home": {"Near": 0, "Moderate": 1, "Far": 2},
}
cols_nominales = ["Gender", "School_Type", "Extracurricular_Activities",
                  "Internet_Access", "Learning_Disabilities", "Peer_Influence"]

df_enc = df_clean.copy()
for col, mapping in mappings_ordinaux.items():
    df_enc[col] = df_enc[col].map(mapping)

df_enc = pd.get_dummies(df_enc, columns=cols_nominales, drop_first=True, dtype=int)
df_enc.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = df_enc.drop(columns="Exam_Score")
y = df_enc["Exam_Score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

cols_num = ["Hours_Studied", "Attendance", "Sleep_Hours",
            "Previous_Scores", "Tutoring_Sessions", "Physical_Activity"]

scaler = ColumnTransformer(
    [("num", StandardScaler(), cols_num)],
    remainder="passthrough",
)

In [ ]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([('scaler', StandardScaler())])

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

modeles = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(random_state=42),
    "SVR": SVR(),
}

pipelines = {
    nom: Pipeline([("model", modele)])
    for nom, modele in modeles.items()
}   

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

pipe_lr = Pipeline([
    ("prep", clone(scaler)),
    ("model", LinearRegression()),
])

pipe_rf = Pipeline([
    ("prep", clone(scaler)),
    ("model", RandomForestRegressor(random_state=42)),
])

pipe_sv = Pipeline([
    ("prep", clone(scaler)),
    ("model", SVR())
])

In [ ]:
import time
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

resultats = []
for nom, pipe in [("LinearRegression", pipe_lr), ("RandomForest", pipe_rf)]:
    debut = time.time()
    pipe.fit(X_train, y_train)
    duree = time.time() - debut

    pred_test = pipe.predict(X_test)
    resultats.append({
        "Modele": nom,
        "RMSE": root_mean_squared_error(y_test, pred_test),
        "MAE": mean_absolute_error(y_test, pred_test),
        "R2 test": r2_score(y_test, pred_test),
        "R2 train": r2_score(y_train, pipe.predict(X_train)),
        "Temps fit (s)": duree,
    })

pd.DataFrame(resultats).set_index("Modele").round(3)

In [ ]:
from sklearn.model_selection import GridSearchCV

grilles_params = {
    "LinearRegression": {
        "model__fit_intercept": [True, False],
        "model__positive": [False, True],
    },
    "RandomForest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_split": [2, 5],
    },
    "SVR": {
        "model__C": [0.1, 1, 10],
        "model__kernel": ["linear", "rbf"],
        "model__epsilon": [0.01, 0.1, 0.5],
    },
}

resultats_grid = {}

def lancer_grille(nom):
    grid = GridSearchCV(
        estimator=pipelines[nom],
        param_grid=grilles_params[nom],
        cv=3, scoring="r2", n_jobs=-1, verbose=1,
    )
    grid.fit(X_train, y_train)
    resultats_grid[nom] = grid
    return grid

In [82]:
lancer_grille("LinearRegression")
lancer_grille("RandomForest")
lancer_grille("SVR")

Fitting 3 folds for each of 4 candidates, totalling 12 fits
LinearRegression : 5 s | meilleur R² CV = 0.9898
Meilleurs paramètres : {'model__fit_intercept': True, 'model__positive': False}
Fitting 3 folds for each of 12 candidates, totalling 36 fits
RandomForest : 10 s | meilleur R² CV = 0.8810
Meilleurs paramètres : {'model__max_depth': 20, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Fitting 3 folds for each of 18 candidates, totalling 54 fits
SVR : 162 s | meilleur R² CV = 0.9898
Meilleurs paramètres : {'model__C': 10, 'model__epsilon': 0.5, 'model__kernel': 'linear'}


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...del', SVR())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.1, 1, ...], 'model__epsilon': [0.01, 0.1, ...], 'model__kernel': ['linear', 'rbf']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plo

In [83]:
from sklearn.base import clone
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

def metriques(modele):
    pred = modele.predict(X_test)
    return {
        "RMSE": root_mean_squared_error(y_test, pred),
        "MAE": mean_absolute_error(y_test, pred),
        "R2": r2_score(y_test, pred),
    }

avant = {nom: metriques(clone(pipelines[nom]).fit(X_train, y_train))
         for nom in grilles_params}
df_avant = pd.DataFrame(avant).T.round(4)
df_avant

,RMSE,MAE,R2
LinearRegression,0.3271,0.2716,0.9897
RandomForest,1.0677,0.8458,0.8905
SVR,1.2176,0.9844,0.8575


In [84]:
best_models = {nom: grid.best_estimator_ for nom, grid in resultats_grid.items()}

df_apres = pd.DataFrame({nom: metriques(m) for nom, m in best_models.items()}).T.round(4)

df_comparaison = df_avant.join(df_apres, lsuffix="_avant", rsuffix="_apres")
df_comparaison.to_csv("../reports/comparaison_avant_apres.csv")
df_comparaison

,RMSE_avant,MAE_avant,R2_avant,RMSE_apres,MAE_apres,R2_apres
LinearRegression,0.3271,0.2716,0.9897,0.3271,0.2716,0.9897
RandomForest,1.0677,0.8458,0.8905,1.0633,0.8441,0.8914
SVR,1.2176,0.9844,0.8575,0.3264,0.2710,0.9898


In [85]:
preds = {nom: m.predict(X_test) for nom, m in best_models.items()}
residus = {nom: y_test - p for nom, p in preds.items()}

In [ ]:
nom_final = df_apres["R2"].idxmax()
modele_final = best_models[nom_final]

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['Hours_Studied','Attendance','Parental_Involvement',..., 'Learning_Disabilities_Yes','Peer_Influence_Neutral', 'Peer_Influence_Positive']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'linear'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",10
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.5
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3


In [89]:
import joblib

joblib.dump(modele_final, "../models/final_pipeline.joblib")
joblib.dump(list(X_train.columns), "../models/columns.joblib")

['../models/columns.joblib']